# Wind Speed Histogram Workflow

This notebook shows the wind speed histogram upload flow as clear notebook steps.

**Conceptual steps**

- Resolve the backend ids for the site, model definition, and workbook scopes.
- Load the workbook sheet and detect the histogram bin columns that start with `[`.
- Parse the bin labels and build one prepared series per title and scope with site and location metadata attached.
- Create or reuse one analysis for the chosen timestamp.
- Create or update the result rows linked to that analysis.
- Read the saved rows back and rebuild the normalized dataframe.
- Plot the persisted results as a histogram view.

**How to run it**

- Run the notebook from top to bottom.
- Set `CREATE_NEW_ANALYSIS = True` to create a new analysis row.
- Set `CREATE_NEW_ANALYSIS = False` to reuse the analysis found for `ANALYSIS_TIMESTAMP`.
- Set `UPLOAD_RESULTS = True` to create or update result rows for the resolved site and location scopes.
- Set `UPLOAD_RESULTS = False` to skip writes and only inspect the selected analysis.

## Mermaid Color Legend

All workflow diagrams use the same color meaning.

- <span style="color:#0B5CAD;"><strong>Blue</strong></span>: API call.
- <span style="color:#2E7D32;"><strong>Green</strong></span>: data we keep or check.
- <span style="color:#A56A00;"><strong>Yellow</strong></span>: data we build or reshape.
- <span style="color:#C04A2F;"><strong>Red</strong></span>: choice or stop condition.
- <span style="color:#4B5563;"><strong>Grey line</strong></span>: step-to-step flow.

*Read each diagram from top to bottom.*

## Imports

This section loads the APIs, serializers, workbook tools, and plotting services used later in the notebook.

*Outcome:* all notebook steps use the same local SDK code from this workspace.

In [14]:
%load_ext autoreload
%autoreload 2

import datetime
from pathlib import Path

import pandas as pd
from IPython.display import display
from owi.metadatabase.geometry.io import GeometryAPI
from owi.metadatabase.locations.io import LocationsAPI
from rich import print

from owi.metadatabase.results import ResultsAPI, WindSpeedHistogram
from owi.metadatabase.results.models import AnalysisDefinition
from owi.metadatabase.results.serializers import DjangoAnalysisSerializer, DjangoResultSerializer
from owi.metadatabase.results.services import ApiResultsRepository, ResultsService
from owi.metadatabase.results.utils import load_token_from_env_file


The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


## Constants

This section sets the workbook path, API root, project metadata, timestamp, and write controls.

**Check before you run**

- `WORKBOOK` points to the Excel file with the wind speed histogram sheet.
- `PROJECTSITE` and `MODEL_DEFINITION` point to the right backend objects.
- `ANALYSIS_TIMESTAMP` picks the analysis row to create or reuse.
- `CREATE_NEW_ANALYSIS` decides whether to create a new analysis.
- `UPLOAD_RESULTS` decides whether result rows are written.

*Outcome:* the notebook has one place to review all runtime choices before any API call.*

<html>
    <style>
        table {
            border: none;
            border-color: transparent;
            border-spacing: 2px;
            border-collapse: separate;
            border-radius: 8px;
            font-family: "Latin Modern Mono";
        }
    </style>
</html>

In [2]:
WORKSPACE_ROOT = Path.cwd().resolve().parent
DEFAULT_WORKBOOK = WORKSPACE_ROOT / "scripts" / "data" / "results-example-data.xlsx"
DEFAULT_ENV_FILE = WORKSPACE_ROOT / ".env"
TOKEN_ENV_VAR = "OWI_METADATABASE_API_TOKEN"
DEFAULT_RESULTS_API_ROOT = "https://owimetadatabase-dev.azurewebsites.net/api/v1"
BASE_URL = DEFAULT_RESULTS_API_ROOT
WORKBOOK = DEFAULT_WORKBOOK
PROJECTSITE = "Belwind"
MODEL_DEFINITION = f"as-designed {PROJECTSITE}"
TOKEN = load_token_from_env_file(DEFAULT_ENV_FILE, TOKEN_ENV_VAR)
ANALYSIS_TIMESTAMP = datetime.datetime(2026, 3, 31, 12, 0, 0)
CREATE_NEW_ANALYSIS = True
UPLOAD_RESULTS = True

display(pd.DataFrame([{"workbook": str(WORKBOOK),
                       "projectsite": PROJECTSITE,
                       "api_root": BASE_URL,
                       "token_available": TOKEN is not None,
                       "analysis_timestamp": ANALYSIS_TIMESTAMP,
                       "create_new_analysis": CREATE_NEW_ANALYSIS,
                       "upload_results": UPLOAD_RESULTS}]).T.rename(columns={0: "value"}))
if TOKEN is None:
    raise ValueError(f"Set {TOKEN_ENV_VAR} in {DEFAULT_ENV_FILE} or export "
                     "it in the environment before running this notebook.")


,value
workbook,/home/pietro.dantuono@24SEA.local/Projects/SMA...
projectsite,Belwind
api_root,https://owimetadatabase-dev.azurewebsites.net/...
token_available,True
analysis_timestamp,2026-03-31 12:00:00
create_new_analysis,True
upload_results,True


## Resolve Project Metadata

Before reading the workbook, the notebook resolves the backend ids it needs.

**What is resolved**

- `site_id`
- `model_definition_id`
- `location_frame`
- `location_title_to_id_map`
- `existing_analysis_id` for the selected `ANALYSIS_TIMESTAMP`

*Outcome:* later cells can work with stable ids instead of repeating name-based lookups.*

```mermaid
%%{init: {'theme': 'base', 'themeVariables': {
  'fontSize': 'small',
  'lineColor': '#4B5563',
  'clusterBkg': '#F8FAFC',
  'clusterBorder': '#CBD5E1'
}}}%%
graph TD
    A["Notebook configuration"] --> B["Call Locations API"]
    A --> C["Call Geometry API"]
    A --> D["Call Results API"]
    B --> E["site_id and location_frame"]
    C --> F["model_definition_id"]
    D --> G["existing_analysis_id for the timestamp"]
    E --> H["location_title_to_id_map"]
    F --> I["Metadata ready"]
    G --> I
    H --> I

    classDef api fill:#CFE0F5,stroke:#0B5CAD,color:#062B5B;
    classDef keep fill:#DCEFD8,stroke:#2E7D32,color:#103816;
    classDef build fill:#F3E3BF,stroke:#A56A00,color:#4A3200;

    class B,C,D api;
    class E,F,G,H,I keep;
    class A build;
```

In [4]:

# ** Class instances for API access and CEIT results management

locations_api = LocationsAPI(api_root=BASE_URL, token=TOKEN)
geometry_api = GeometryAPI(api_root=BASE_URL, token=TOKEN)
results_api = ResultsAPI(api_root=BASE_URL, token=TOKEN)
analysis = WindSpeedHistogram()
analysis_serializer = DjangoAnalysisSerializer()
results_service = ResultsService(repository=ApiResultsRepository(api=results_api))
result_serializer = DjangoResultSerializer()

# -- site_id
site_id = locations_api.get_projectsite_detail(projectsite=PROJECTSITE)["id"]

# -- model_definition_id
model_definition_id = geometry_api.get_modeldefinition_id(
    projectsite=PROJECTSITE, model_definition=MODEL_DEFINITION
)["id"]

# -- location_title_to_id_map
assetlocations = locations_api.get_assetlocations(projectsite=PROJECTSITE)["data"]
location_frame = assetlocations.loc[
    :, [column for column in ["id", "title", "northing", "easting"]
        if column in assetlocations.columns]
].copy()
location_title_to_id_map = {str(row["title"]): int(row["id"])
                            for row in location_frame.to_dict(orient="records")
                            if row.get("title") is not None and row.get("id") is not None}


# -- analysis_id (if existing)
existing_analysis = results_api.get_analysis(
    name=analysis.analysis_name,
    model_definition__id=model_definition_id,
    timestamp=ANALYSIS_TIMESTAMP,
    location__id=None,
)
existing_analysis_id = (
    None if not existing_analysis["exists"] or existing_analysis["id"] is None else int(existing_analysis["id"])
)
# locations_api.plot_assetlocations(projectsite=PROJECTSITE)
display(pd.DataFrame([{"site": PROJECTSITE,
                       "site_id": site_id,
                       "model_definition": MODEL_DEFINITION,
                       "model_definition_id": model_definition_id,
                       "num_locations": len(location_title_to_id_map),
                       "analysis_id": existing_analysis_id}]).T
                     .reset_index().rename(columns={"index": "metric", 0: "value"})
                     .set_index("metric"))


,value
metric,
site,Belwind
site_id,35
model_definition,as-designed Belwind
model_definition_id,4
num_locations,55
analysis_id,None


## Load and Inspect the Workbook Data

This stage reads the Excel sheet and identifies the histogram bin columns that will feed the analysis workflow.

**What happens here**

- Read the `Lifetime - Wind Histogram` sheet from the workbook.
- Build `histogram_frame` as the raw input table.
- Detect the histogram bin columns that start with `[`.
- Preview the workbook data before analysis and upload logic starts.

*Outcome:* the raw workbook data and bin-column list are ready for edge parsing and row preparation.*

```mermaid
%%{init: {'theme': 'base', 'themeVariables': {
  'fontSize': 'small',
  'lineColor': '#4B5563',
  'clusterBkg': '#F8FAFC',
  'clusterBorder': '#CBD5E1'
}}}%%
graph TD
    A["Workbook sheet"] --> B["Read Excel data"]
    B --> C["histogram_frame"]
    C --> D["Detect histogram bin columns"]
    D --> E["bin_columns"]
    C --> F["Workbook preview"]
    E --> G["Workbook data ready"]
    F --> G

    classDef api fill:#CFE0F5,stroke:#0B5CAD,color:#062B5B;
    classDef keep fill:#DCEFD8,stroke:#2E7D32,color:#103816;
    classDef build fill:#F3E3BF,stroke:#A56A00,color:#4A3200;

    class B api;
    class C,E,F,G keep;
    class A,D build;
```

In [11]:
def _parse_bin_edges(label: str) -> tuple[float, float]:
    """Extract the left and right edges from a histogram bin label like '[0,1['."""
    import re
    numbers = [float(value) for value in re.findall(r"-?\d+(?:\.\d+)?", label)]
    if len(numbers) < 2:
        raise ValueError(f"Could not parse histogram bin edges from {label!r}.")
    return numbers[0], numbers[1]


# -- Actual data processing and upload logic starts here --
sheet_name = "Lifetime - Wind Histogram"
histogram_frame = pd.read_excel(WORKBOOK, sheet_name=sheet_name, header=1)
bin_columns = [column for column in histogram_frame.columns
               if isinstance(column, str) and column.startswith("[")]
print("\033[95mInput data (histogram frame) loaded from Excel:\033[0m")
display(histogram_frame.head())


Input data (histogram frame) loaded from Excel:


,Title,Description,Scope,"[0,1[","[1,2[","[2,3[","[3,4[","[4,5[","[5,6[","[6,7[",...,"[40,41[","[41,42[","[42,43[","[43,44[","[44,45[","[45,46[","[46,47[","[47,48[","[48,49[","[49,50["
0,Design,As-designed windspeed distribution as specifie...,Site,1,2,3,4,5.0,6.0,7.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0
1,WTG1 - Y1,As measured distribution for location WTG1 - Y1,WTG1,50,49,48,47,46.0,45.0,44.0,...,10.0,9.0,8.0,7.0,6.0,5.0,4.0,3.0,2.0,1
2,WTG2 - Y1,As measured distribution for location WTG2 - Y1,WTG2,1,1,2,3,3.5,4.2,4.9,...,28.7,29.4,30.1,30.8,31.5,32.2,32.9,33.6,34.3,35
3,WTG1 - Y2,As measured distribution for location WTG1 - Y1,WTG1,50,49,48,47,46.0,45.0,44.0,...,10.0,9.0,8.0,7.0,6.0,5.0,4.0,3.0,2.0,1
4,WTG2 - Y2,As measured distribution for location WTG2 - Y1,WTG2,1,1,2,3,3.5,4.2,4.9,...,28.7,29.4,30.1,30.8,31.5,32.2,32.9,33.6,34.3,35


## Build and Upload the Shared Analysis

This section builds the shared analysis payload, prepares the workbook rows as typed result series, serializes the payloads, and persists them through `ResultsAPI`.

**Execution logic**

- Build the shared `AnalysisDefinition` with `ANALYSIS_TIMESTAMP` so the backend lookup is unique.
- When `CREATE_NEW_ANALYSIS` is `True`, create a new analysis row and fail fast if the timestamped analysis already exists.
- When `CREATE_NEW_ANALYSIS` is `False`, reuse the existing timestamped analysis row.
- Parse the histogram bin labels into numeric left and right edges before building the prepared series.
- Convert the workbook rows into `WindSpeedHistogram` result series with site metadata and scope-dependent location ids attached.
- Serialize those series into backend payloads with the generic Django serializer.
- When `UPLOAD_RESULTS` is `True`, bulk create or update the result rows for the selected analysis.
- When `UPLOAD_RESULTS` is `False`, skip writes and continue with retrieval against the selected analysis id.

*Outcome:* one shared analysis stores the workbook upload while analysis creation and row upload stay independently controllable.*

```mermaid
%%{init: {'theme': 'base', 'themeVariables': {
  'fontSize': 'small',
  'lineColor': '#4B5563',
  'clusterBkg': '#F8FAFC',
  'clusterBorder': '#CBD5E1'
}}}%%
graph TD
    A["Prepared workbook data and metadata"] --> B{"Create new analysis?"}
    B -- yes --> C["Create the analysis row"]
    B -- no --> D["Reuse the existing analysis id"]
    C --> E["Selected analysis id"]
    D --> E
    E --> F["Parse bins and build prepared_rows"]
    F --> G["Create result_series"]
    G --> H["Serialize result payloads"]
    H --> I{"Upload result rows?"}
    I -- yes --> J["Bulk create or update histogram rows"]
    I -- no --> K["Skip writes"]
    J --> L["Upload summary"]
    K --> L

    classDef api fill:#CFE0F5,stroke:#0B5CAD,color:#062B5B;
    classDef keep fill:#DCEFD8,stroke:#2E7D32,color:#103816;
    classDef build fill:#F3E3BF,stroke:#A56A00,color:#4A3200;
    classDef decision fill:#F7D9D1,stroke:#C04A2F,color:#5A1F14;

    class C,J api;
    class D,E,L keep;
    class A,F,G,H,K build;
    class B,I decision;
```

In [ ]:

# ** Create analysis (if configured to do so and if not already existing)

analysis_definition = AnalysisDefinition(
    name=analysis.analysis_name,
    model_definition_id=model_definition_id,
    location_id=None,
    source_type="json",
    source=str(WORKBOOK),
    timestamp=ANALYSIS_TIMESTAMP,
    description="Shared CEIT corrosion monitoring upload.",
    additional_data={"input_file": WORKBOOK.name, "sheet_name": sheet_name}
)
analysis_payload = analysis_serializer.to_payload(analysis_definition)

if CREATE_NEW_ANALYSIS and existing_analysis_id is not None:
    raise ValueError(
        "An analysis already exists for the configured name, timestamp, model definition, and location. "
        "Set CREATE_NEW_ANALYSIS to False or choose a different ANALYSIS_TIMESTAMP."
    )

if CREATE_NEW_ANALYSIS:
    created_analysis = results_api.create_analysis(analysis_payload)
    analysis_id = int(created_analysis["id"])
    analysis_created = True
else:
    if existing_analysis_id is None:
        raise ValueError("No existing analysis matched the configured name, timestamp, "
                         "model definition, and location.\nSet CREATE_NEW_ANALYSIS to True "
                         "or adjust ANALYSIS_TIMESTAMP.")
    analysis_id = existing_analysis_id
    analysis_created = False


In [12]:
display(pd.DataFrame({"bin_column": bin_columns[:10],
                       "parsed_edges": [_parse_bin_edges(c) for c in bin_columns[:10]]}))
prepared_rows = [{"title": str(row["Title"]),
                  "description": row.get("Description"),
                  "scope_label": str(row["Scope"]).strip(),
                  "site_id": site_id,
                  "location_id": (None if str(row["Scope"]).strip() == "Site"
                                  else location_title_to_id_map.get(str(row["Scope"]).strip())),
                  "bins": [_parse_bin_edges(str(column)) for column in bin_columns],
                  "values": [float(row[column]) for column in bin_columns],
                  "metadata": {"sheet_scope": str(row["Scope"]).strip()}}
                 for row in histogram_frame.to_dict(orient="records")]
# analysis.to_results performs the validation step of the prepared series
# against the result model, and converts them into a list of result
# objects. This step is crucial to ensure that the data conforms to the
# expected structure and types before serialization and upload.
result_series = analysis.to_results({"series": prepared_rows})


,bin_column,parsed_edges
0,"[0,1[","(0.0, 1.0)"
1,"[1,2[","(1.0, 2.0)"
2,"[2,3[","(2.0, 3.0)"
3,"[3,4[","(3.0, 4.0)"
4,"[4,5[","(4.0, 5.0)"
5,"[5,6[","(5.0, 6.0)"
6,"[6,7[","(6.0, 7.0)"
7,"[7,8[","(7.0, 8.0)"
8,"[8,9[","(8.0, 9.0)"
9,"[9,10[","(9.0, 10.0)"


In [13]:

# ** Upload results to the API (if configured to do so)

results_payloads = [result_serializer.to_payload(series, analysis_id=analysis_id)
                   for series in result_series]
upload_result = {"summary": [], "data": pd.DataFrame(), "exists": False}
if UPLOAD_RESULTS:
    upload_result = results_api.create_or_update_results_bulk(results_payloads)

# --- Sumary

summary_frame = pd.DataFrame(upload_result["summary"],
                             columns=["analysis", "short_description", "result_id", "action"])
payload_summary = pd.DataFrame([{"analysis": int(payload["analysis"]),
                                 "short_description": str(payload["short_description"])}
                                for payload in results_payloads])
upload_summary = payload_summary.merge(summary_frame,
                                       on=["analysis", "short_description"],
                                       how="left")
upload_summary = payload_summary.merge(summary_frame,
                                       on=["analysis", "short_description"],
                                       how="left")
if not upload_summary.empty:
    upload_summary = upload_summary.rename(columns={"short_description": "series_key"})

print("[bold green]Preview of analysis and results payloads[/bold green]")
display(pd.DataFrame([analysis_payload]))
print("[blue]• Sample of results payloads for the first sensor[/blue]")
display(pd.DataFrame(results_payloads[:5]))
print("[blue]• Upload summary[/blue]")
display(upload_summary)
print("[blue]• Summary of uploaded series by sensor and metric[/blue]")
display(pd.DataFrame([{"analysis_id": analysis_id,
                       "analysis_created": analysis_created,
                       "analysis_timestamp": ANALYSIS_TIMESTAMP,
                       "upload_results": UPLOAD_RESULTS,
                       "series_uploaded": len(upload_summary)}]) \
                     .T.rename(columns={0: "value"}))


Uploading result batch:   0%|          | 0/1 [00:00<?, ?request/s]

Uploading result rows:   0%|          | 0/5 [00:00<?, ?row/s]

[bold green]Preview of analysis and results payloads[/bold green]


,name,model_definition_id,source_type,source,description,timestamp,additional_data
0,WindSpeedHistogram,4,json,/home/pietro.dantuono@24SEA.local/Projects/SMA...,Shared CEIT corrosion monitoring upload.,2026-03-31 12:00:00,"{'input_file': 'results-example-data.xlsx', 's..."


[blue]• Sample of results payloads for the first sensor[/blue]


,analysis,site,name_col1,name_col2,name_col3,units_col1,units_col2,units_col3,value_col1,value_col2,value_col3,short_description,description,additional_data
0,75,35,bin_left,value,bin_right,m/s,count,m/s,"[0.0, 1.0, 2.0, 3.0, 4.0, 5.0, 6.0, 7.0, 8.0, ...","[1.0, 2.0, 3.0, 4.0, 5.0, 6.0, 7.0, 8.0, 9.0, ...","[1.0, 2.0, 3.0, 4.0, 5.0, 6.0, 7.0, 8.0, 9.0, ...",Design,As-designed windspeed distribution as specifie...,"{'analysis_kind': 'histogram', 'result_scope':..."
1,75,35,bin_left,value,bin_right,m/s,count,m/s,"[0.0, 1.0, 2.0, 3.0, 4.0, 5.0, 6.0, 7.0, 8.0, ...","[50.0, 49.0, 48.0, 47.0, 46.0, 45.0, 44.0, 43....","[1.0, 2.0, 3.0, 4.0, 5.0, 6.0, 7.0, 8.0, 9.0, ...",WTG1 - Y1,As measured distribution for location WTG1 - Y1,"{'analysis_kind': 'histogram', 'result_scope':..."
2,75,35,bin_left,value,bin_right,m/s,count,m/s,"[0.0, 1.0, 2.0, 3.0, 4.0, 5.0, 6.0, 7.0, 8.0, ...","[1.0, 1.0, 2.0, 3.0, 3.5, 4.2, 4.9, 5.6, 6.3, ...","[1.0, 2.0, 3.0, 4.0, 5.0, 6.0, 7.0, 8.0, 9.0, ...",WTG2 - Y1,As measured distribution for location WTG2 - Y1,"{'analysis_kind': 'histogram', 'result_scope':..."
3,75,35,bin_left,value,bin_right,m/s,count,m/s,"[0.0, 1.0, 2.0, 3.0, 4.0, 5.0, 6.0, 7.0, 8.0, ...","[50.0, 49.0, 48.0, 47.0, 46.0, 45.0, 44.0, 43....","[1.0, 2.0, 3.0, 4.0, 5.0, 6.0, 7.0, 8.0, 9.0, ...",WTG1 - Y2,As measured distribution for location WTG1 - Y1,"{'analysis_kind': 'histogram', 'result_scope':..."
4,75,35,bin_left,value,bin_right,m/s,count,m/s,"[0.0, 1.0, 2.0, 3.0, 4.0, 5.0, 6.0, 7.0, 8.0, ...","[1.0, 1.0, 2.0, 3.0, 3.5, 4.2, 4.9, 5.6, 6.3, ...","[1.0, 2.0, 3.0, 4.0, 5.0, 6.0, 7.0, 8.0, 9.0, ...",WTG2 - Y2,As measured distribution for location WTG2 - Y1,"{'analysis_kind': 'histogram', 'result_scope':..."


[blue]• Upload summary[/blue]


,analysis,series_key,result_id,action
0,75,Design,6564,created
1,75,WTG1 - Y1,6565,created
2,75,WTG2 - Y1,6566,created
3,75,WTG1 - Y2,6567,created
4,75,WTG2 - Y2,6568,created


[blue]• Summary of uploaded series by sensor and metric[/blue]


,value
analysis_id,75
analysis_created,True
analysis_timestamp,2026-03-31 12:00:00
upload_results,True
series_uploaded,5


## Retrieve the Persisted Results

This stage reads the saved rows back from the API and rebuilds the wind speed histogram dataframe from them.

**What happens here**

- Read the raw result rows for the selected analysis.
- Convert those rows back into result series objects.
- Rebuild the normalized wind speed histogram dataframe from the saved series.

*Outcome:* the notebook checks the saved backend data, not just the original workbook input.*

```mermaid
%%{init: {'theme': 'base', 'themeVariables': {
  'fontSize': 'small',
  'lineColor': '#4B5563',
  'clusterBkg': '#F8FAFC',
  'clusterBorder': '#CBD5E1'
}}}%%
graph TD
    A["Selected analysis id"] --> B["Read saved result rows"]
    B --> C["Raw backend table"]
    C --> D["Convert rows back to result series"]
    D --> E["Rebuild the histogram dataframe"]
    E --> F["Retrieved dataframe ready"]

    classDef api fill:#CFE0F5,stroke:#0B5CAD,color:#062B5B;
    classDef keep fill:#DCEFD8,stroke:#2E7D32,color:#103816;
    classDef build fill:#F3E3BF,stroke:#A56A00,color:#4A3200;

    class B api;
    class C,F keep;
    class A,D,E build;
```

In [18]:
# Use the analysis_id obtained above, either from the creation step or from the
# existing analysis query.
raw_results_frame = results_api.list_results(analysis=analysis_id)["data"]
print("[bold magenta]Raw results frame retrieved from API:[/bold magenta]")
display(raw_results_frame.head())
retrieved_series = [result_serializer.from_mapping(row)
                    for row in raw_results_frame.to_dict(orient="records")]
retrieved_frame = analysis.from_results(retrieved_series)
print("[green]Retrieved results frame reconstructed from API data:[/green]")
display(retrieved_frame.head())


Raw results frame retrieved from API:

,id,name_col1,name_col2,name_col3,units_col1,units_col2,units_col3,value_col1,value_col2,value_col3,analysis,site,location,short_description,description,related_object,additional_data,slug,visibility,visibility_groups
0,6564,bin_left,value,bin_right,m/s,count,m/s,"[0.0, 1.0, 2.0, 3.0, 4.0, 5.0, 6.0, 7.0, 8.0, ...","[1.0, 2.0, 3.0, 4.0, 5.0, 6.0, 7.0, 8.0, 9.0, ...","[1.0, 2.0, 3.0, 4.0, 5.0, 6.0, 7.0, 8.0, 9.0, ...",75,35,None,Design,As-designed windspeed distribution as specifie...,None,"{'series_key': 'Design', 'scope_label': 'Site'...",belwind_none_windspeedhistogram_75_design_6564,usergroup,[10]
1,6565,bin_left,value,bin_right,m/s,count,m/s,"[0.0, 1.0, 2.0, 3.0, 4.0, 5.0, 6.0, 7.0, 8.0, ...","[50.0, 49.0, 48.0, 47.0, 46.0, 45.0, 44.0, 43....","[1.0, 2.0, 3.0, 4.0, 5.0, 6.0, 7.0, 8.0, 9.0, ...",75,35,None,WTG1 - Y1,As measured distribution for location WTG1 - Y1,None,"{'series_key': 'WTG1 - Y1', 'scope_label': 'WT...",belwind_none_windspeedhistogram_75_wtg1-y1_6565,usergroup,[10]
2,6567,bin_left,value,bin_right,m/s,count,m/s,"[0.0, 1.0, 2.0, 3.0, 4.0, 5.0, 6.0, 7.0, 8.0, ...","[50.0, 49.0, 48.0, 47.0, 46.0, 45.0, 44.0, 43....","[1.0, 2.0, 3.0, 4.0, 5.0, 6.0, 7.0, 8.0, 9.0, ...",75,35,None,WTG1 - Y2,As measured distribution for location WTG1 - Y1,None,"{'series_key': 'WTG1 - Y2', 'scope_label': 'WT...",belwind_none_windspeedhistogram_75_wtg1-y2_6567,usergroup,[10]
3,6566,bin_left,value,bin_right,m/s,count,m/s,"[0.0, 1.0, 2.0, 3.0, 4.0, 5.0, 6.0, 7.0, 8.0, ...","[1.0, 1.0, 2.0, 3.0, 3.5, 4.2, 4.9, 5.6, 6.3, ...","[1.0, 2.0, 3.0, 4.0, 5.0, 6.0, 7.0, 8.0, 9.0, ...",75,35,None,WTG2 - Y1,As measured distribution for location WTG2 - Y1,None,"{'series_key': 'WTG2 - Y1', 'scope_label': 'WT...",belwind_none_windspeedhistogram_75_wtg2-y1_6566,usergroup,[10]
4,6568,bin_left,value,bin_right,m/s,count,m/s,"[0.0, 1.0, 2.0, 3.0, 4.0, 5.0, 6.0, 7.0, 8.0, ...","[1.0, 1.0, 2.0, 3.0, 3.5, 4.2, 4.9, 5.6, 6.3, ...","[1.0, 2.0, 3.0, 4.0, 5.0, 6.0, 7.0, 8.0, 9.0, ...",75,35,None,WTG2 - Y2,As measured distribution for location WTG2 - Y1,None,"{'series_key': 'WTG2 - Y2', 'scope_label': 'WT...",belwind_none_windspeedhistogram_75_wtg2-y2_6568,usergroup,[10]


Retrieved results frame reconstructed from API data:

,series_name,scope,bin_left,bin_right,value
0,Design,Site,0.0,1.0,1.0
1,Design,Site,1.0,2.0,2.0
2,Design,Site,2.0,3.0,3.0
3,Design,Site,3.0,4.0,4.0
4,Design,Site,4.0,5.0,5.0


## Plot the Retrieved Results

The final stage renders the wind speed histogram results from the retrieved backend rows, not from the raw workbook file.

**What to inspect in the plot**

- The histogram plot should preserve the uploaded bin structure for each series.
- The plotted series should reflect the site or turbine scope stored in the retrieved dataframe.
- The plotted values should match the bin edges and counts shown in the retrieved dataframe preview.

*Outcome:* the same persisted analysis can be inspected through the histogram plotting path exposed by the SDK.*

```mermaid
%%{init: {'theme': 'base', 'themeVariables': {
  'fontSize': 'small',
  'lineColor': '#4B5563',
  'clusterBkg': '#F8FAFC',
  'clusterBorder': '#CBD5E1'
}}}%%
graph TD
    A["Selected analysis id"] --> B["Create histogram plot"]
    B --> C["Histogram widget"]
    C --> D["Display plot"]

    classDef api fill:#CFE0F5,stroke:#0B5CAD,color:#062B5B;
    classDef keep fill:#DCEFD8,stroke:#2E7D32,color:#103816;
    classDef build fill:#F3E3BF,stroke:#A56A00,color:#4A3200;

    class B api;
    class C,D keep;
    class A build;
```

In [19]:

filters = {"analysis_id": analysis_id}

histogram_plot = results_service.plot_results(
    analysis.analysis_name,
    filters=filters,
    plot_type="histogram",
)

display(histogram_plot.notebook)


In [20]:
plot_summary = pd.DataFrame([{"analysis_id": analysis_id,
                              "retrieved_rows": len(raw_results_frame),
                              "normalized_rows": len(retrieved_frame),
                              "series_count": len(retrieved_series),
                              "histogram_plot_available": histogram_plot.notebook is not None}]) \
                            .T.reset_index().rename(columns={"index": "metric", 0: "value"}) \
                            .set_index("metric")

display(plot_summary)


,value
metric,
analysis_id,75
retrieved_rows,5
normalized_rows,250
series_count,5
histogram_plot_available,True
